# CHART MVP climate data explorer

Short notebook for checking the climate-data shape before wiring it into CHART.

Default mode uses real Copernicus ERA5 reanalysis for the Sprint 4 MVP window, 2020-2024. Set `USE_FIXTURE = True` only for offline preview or tests.

In [ ]:
from pathlib import Path
import os
import sys

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pipelines" / "era5_heat" / "src").exists()
)
os.environ.setdefault(
    "MPLCONFIGDIR",
    str(project_root / "pipelines" / "era5_heat" / ".cache" / "matplotlib"),
)
sys.path.insert(0, str(project_root / "pipelines" / "era5_heat" / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from era5_heat import PRESETS, compute_heat_series, fixture_demo

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 30)

## 1. Choose the data slice

For Sprint 4, keep this narrow: Madhya Pradesh and Kajiado, historic monthly heat metrics, no forecasting yet.

In [ ]:
USE_FIXTURE = False
PRESET_SLUGS = ["madhya-pradesh", "kajiado"]
YEARS = 5
END_YEAR = 2024
THRESHOLD_C = 35.0
MIN_RUN = 3

for slug in PRESET_SLUGS:
    preset = PRESETS[slug]
    print(f"{slug}: {preset.name}, {preset.country}, bbox={preset.bbox}")

## 2. Load monthly heat metrics

Output columns expected by the MVP:

- `district`
- `month`
- `tmax_monthly_max_c`
- `tmax_monthly_mean_c`
- `heatwave_days`
- `observed_days`, `expected_days`, `completeness_pct`, `quality_flag`
- `climate_source`, `climate_dataset`, `data_status`, `generated_at`
- `window_start_year`, `window_end_year`, `threshold_c`, `min_run`

In [ ]:
frames = []
metadata = {}

for slug in PRESET_SLUGS:
    if USE_FIXTURE:
        df, meta = fixture_demo(
            slug,
            years=YEARS,
            end_year=END_YEAR,
            threshold_c=THRESHOLD_C,
            min_run=MIN_RUN,
        )
    else:
        preset = PRESETS[slug]
        df, meta = compute_heat_series(
            district=preset.name,
            bbox=preset.bbox,
            years=YEARS,
            end_year=END_YEAR,
            threshold_c=THRESHOLD_C,
            min_run=MIN_RUN,
        )

    df = df.copy()
    df["preset"] = slug
    df["month"] = pd.to_datetime(df["month"])
    df["year"] = df["month"].dt.year
    df["month_num"] = df["month"].dt.month
    frames.append(df)
    metadata[slug] = meta

climate = pd.concat(frames, ignore_index=True)
climate.head()

## 3. Sanity checks

These are the checks that matter before the data is used in a dashboard or model handoff.

In [ ]:
checks = {
    "rows": len(climate),
    "missing_values": int(climate.isna().sum().sum()),
    "max_less_than_mean_rows": int((climate["tmax_monthly_max_c"] < climate["tmax_monthly_mean_c"]).sum()),
    "negative_heatwave_day_rows": int((climate["heatwave_days"] < 0).sum()),
    "heatwave_days_over_31_rows": int((climate["heatwave_days"] > 31).sum()),
    "partial_quality_rows": int((climate["quality_flag"] != "complete").sum()),
    "status_values": sorted(climate["data_status"].unique().tolist()),
}
checks

In [ ]:
climate.groupby("district")[["tmax_monthly_max_c", "tmax_monthly_mean_c", "heatwave_days"]].agg(
    ["min", "mean", "max"]
).round(2)

## 4. Quick plots

Use these to explain the data quickly: seasonal pattern, annual heatwave burden, and hottest months.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (district, sub) in zip(axes, climate.groupby("district")):
    monthly = sub.groupby("month_num")["tmax_monthly_max_c"].median()
    monthly.plot(ax=ax, marker="o", color="#b33a2b")
    ax.axhline(THRESHOLD_C, color="#333333", linestyle="--", linewidth=1, label=f"{THRESHOLD_C:g} C")
    ax.set_title(district)
    ax.set_xlabel("Month")
    ax.set_ylabel("Median monthly max Tmax (C)")
    ax.legend()
fig.tight_layout()

In [ ]:
annual = climate.groupby(["district", "year"], as_index=False).agg(
    annual_peak_tmax_c=("tmax_monthly_max_c", "max"),
    annual_heatwave_days=("heatwave_days", "sum"),
    annual_min_completeness=("completeness_pct", "min"),
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for district, sub in annual.groupby("district"):
    axes[0].plot(sub["year"], sub["annual_peak_tmax_c"], marker="o", label=district)
    axes[1].plot(sub["year"], sub["annual_heatwave_days"], marker="o", label=district)
axes[0].set_title("Annual peak temperature")
axes[0].set_ylabel("C")
axes[1].set_title("Annual heatwave days")
axes[1].set_ylabel("Days")
for ax in axes:
    ax.set_xlabel("Year")
    ax.legend()
fig.tight_layout()

In [ ]:
climate.sort_values("tmax_monthly_max_c", ascending=False)[
    ["district", "year", "month_num", "tmax_monthly_max_c", "heatwave_days"]
].head(12)

## 5. MVP handoff shape

This is the small contract CHART can consume before a full modeling pipeline exists.

In [ ]:
mvp_handoff = climate[[
    "district",
    "month",
    "tmax_monthly_max_c",
    "tmax_monthly_mean_c",
    "heatwave_days",
    "observed_days",
    "expected_days",
    "completeness_pct",
    "quality_flag",
    "threshold_c",
    "min_run",
    "climate_source",
    "climate_dataset",
    "climate_source_version",
    "climate_variable",
    "data_status",
    "generated_at",
    "window_start_year",
    "window_end_year",
]]

mvp_handoff.head()